# 🤖 LangGraph Assistants 概念详解

> 从这节课你能学到什么？如何用 Assistant 实现「同一个 graph，不同配置」的多角色系统

---

## 📚 这节课教了什么？

这节课的核心概念是 **Assistant**（助手）。

一句话总结：

> **Assistant = 同一个 Graph + 不同的 Configuration**

你部署了一个 `task_maistro` graph，但你可以创建多个 assistant，每个 assistant 有不同的配置（角色、分类、用户 ID 等）。

---

## 🧠 核心概念

### Assistant 是什么？

| 概念 | 类比 |
|------|------|
| Graph | 一个 App 的代码 |
| Assistant | 这个 App 的不同「用户配置」 |

比如你写了一个 ToDo 管理 agent（graph），你可以创建：

- 🏠 **Personal Assistant** — 管理个人任务，语气友好温暖
- 💼 **Work Assistant** — 管理工作任务，语气专业高效

它们共享同一个 graph 逻辑，但行为不同，因为 **configuration 不同**。

---

## 🔧 Configuration 的三个可配置字段

在 `configuration.py` 中定义：

| 字段 | 作用 | 示例 |
|------|------|------|
| `user_id` | 标识用户，用于 memory store 的 namespace | `"lance"` |
| `todo_category` | 任务分类，决定存储在哪个 namespace | `"personal"` / `"work"` |
| `task_maistro_role` | System prompt，定义 agent 的角色和行为 | 一段详细的角色描述 |

这些字段在 graph 的 node 函数中通过 `RunnableConfig` 读取。

---

## 🛠️ SDK 操作 Assistant 的 API

```python
from langgraph_sdk import get_client
client = get_client(url="http://localhost:8123")
```

| 操作 | 代码 | 说明 |
|------|------|------|
| 创建 | `client.assistants.create(graph_id, config={...})` | 基于 graph 创建一个新 assistant |
| 更新 | `client.assistants.update(assistant_id, config={...})` | 修改配置，版本号 +1 |
| 搜索 | `client.assistants.search()` | 列出所有 assistant |
| 删除 | `client.assistants.delete(assistant_id)` | 删除一个 assistant |

### ⚡ 版本管理

每次 `update` 一个 assistant，版本号会自动递增（v1 → v2 → v3...）。
这让你可以实验不同的 prompt/配置，随时回滚。

---

## 🎯 实际使用流程

```text
1. 部署 graph（docker compose up）
        ↓
2. 用 SDK 创建 assistant（指定 config）
        ↓
3. 创建 thread
        ↓
4. 用 assistant_id 执行 run
        ↓
5. Graph 内部读取 config，按配置行为
```

关键代码：

```python
# 创建 thread
thread = await client.threads.create()

# 用 work assistant 执行
async for chunk in client.runs.stream(
    thread["thread_id"],
    work_assistant_id,  # ← 这里指定用哪个 assistant
    input={"messages": [HumanMessage(content="...")]},
    stream_mode="values"
):
    ...
```

---

## 💡 关键洞察

### 1. Assistant ≠ Thread

- **Assistant** = 配置（角色、分类）— 持久存在
- **Thread** = 一次对话 — 可以创建多个

同一个 assistant 可以有多个 thread（多次对话）。

### 2. Memory 隔离靠 `todo_category`

Personal assistant 的任务存在 `("lance", "personal")` namespace。
Work assistant 的任务存在 `("lance", "work")` namespace。

所以「给我 todo summary」时，personal assistant 只看到个人任务。

### 3. 行为差异靠 `task_maistro_role`

- Personal: 语气温暖，鼓励性
- Work: 语气专业，会根据任务类型建议时间估算

### 4. Assistant 存在 Postgres 中

部署后，assistant 的配置持久化在数据库里，不会因为重启丢失。

---

## 🗺️ 在整个 LangGraph 生态中的位置

```text
LangGraph (graph 逻辑)
    ↓
LangGraph API (运行 graph 的 server)
    ↓
Assistant (graph + config 的组合) ← 📍 这节课
    ↓
Thread (一次对话)
    ↓
Run (一次执行)
```

Assistant 是 API 层面的概念，让你不用改代码就能创建不同行为的 agent 实例。